# Model Performance Test
Quick sanity-check for a fine-tuned ResNet-50 checkpoint on the MVTec AD test split.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CATEGORY = "carpet"   # change to any of: carpet, bottle, metal_nut, transistor, leather
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
WORK_PYTHON_CANDIDATES = [
    ROOT,
    ROOT / "work" / "python",
    ROOT.parent / "work" / "python",
    ROOT.parent.parent / "work" / "python",
]
WORK_PYTHON = next((p for p in WORK_PYTHON_CANDIDATES if (p / "config.py").exists()), None)
if WORK_PYTHON is None:
    raise RuntimeError("Could not locate work/python (config.py). Open notebook from repo root or work/python.")
if str(WORK_PYTHON) not in sys.path:
    sys.path.insert(0, str(WORK_PYTHON))

from config import CHECKPOINT_DIR, MVTEC_ROOT
from data.mvtec import MVTecDataset, get_transforms
from models.resnet import load_checkpoint


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt_path = CHECKPOINT_DIR / f"resnet50_{CATEGORY}.pth"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path} — run train.py first"

model, train_acc = load_checkpoint(ckpt_path, device)
model.eval()
print(f"Loaded: {ckpt_path.name}  |  recorded train_acc = {train_acc:.4f}")

In [ ]:
tf = get_transforms(augment=False)
test_ds = MVTecDataset(MVTEC_ROOT, CATEGORY, split="test", transform=tf)

# num_workers=0 is the safest default in notebook kernels (avoids multiprocessing hangs)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
print(f"Test samples: {len(test_ds)}")


In [ ]:
all_labels, all_preds, all_scores, all_paths = [], [], [], []

with torch.no_grad():
    for imgs, labels, paths in tqdm(test_loader, desc="Evaluating"):
        logits = model(imgs.to(device))
        probs  = F.softmax(logits, dim=1).cpu()
        preds  = probs.argmax(dim=1).tolist()
        scores = probs[:, 1].tolist()   # defective-class confidence

        all_labels.extend(labels.tolist())
        all_preds.extend(preds)
        all_scores.extend(scores)
        all_paths.extend(paths)

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_scores = np.array(all_scores)

In [ ]:
# Diagnostic: raw prediction counts
import pandas as pd
df = pd.DataFrame({"true": all_labels, "pred": all_preds})
print("Prediction breakdown:")
print(df.groupby(["true","pred"]).size().rename("count").reset_index()
        .replace({0:"normal",1:"defective"})
        .to_string(index=False))
n_pred_normal    = (all_preds == 0).sum()
n_pred_defective = (all_preds == 1).sum()
print(f"\nPredicted normal: {n_pred_normal}  |  Predicted defective: {n_pred_defective}")

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
acc  = (all_labels == all_preds).mean()
auroc = roc_auc_score(all_labels, all_scores)

print(f"Accuracy : {acc:.4f}")
print(f"AUROC    : {auroc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["normal", "defective"]))

In [ ]:
# ── Confusion matrix + ROC curve ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"{CATEGORY}  —  test-set performance", fontsize=13)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=["normal", "defective"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_scores)
axes[1].plot(fpr, tpr, lw=2, label=f"AUROC = {auroc:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC curve")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Score distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_scores[all_labels == 0], bins=40, alpha=0.6, label="normal",    color="steelblue")
ax.hist(all_scores[all_labels == 1], bins=40, alpha=0.6, label="defective", color="tomato")
ax.set_xlabel("Defective-class confidence"); ax.set_ylabel("Count")
ax.set_title(f"{CATEGORY}  —  score distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Sample predictions (8 correct + 8 wrong) ─────────────────────────────────
from PIL import Image as PILImage

def _show_grid(indices, title):
    n = len(indices)
    cols = min(n, 8)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.2))
    axes = np.array(axes).reshape(-1)  # flatten for easy indexing
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, indices):
        img = PILImage.open(all_paths[i]).convert("RGB").resize((112, 112))
        true_str = "normal" if all_labels[i] == 0 else "defect"
        pred_str = "normal" if all_preds[i]  == 0 else "defect"
        ax.imshow(img)
        ax.set_title(f"T:{true_str}\nP:{pred_str} ({all_scores[i]:.2f})", fontsize=7)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

correct = np.where(all_labels == all_preds)[0]
wrong   = np.where(all_labels != all_preds)[0]

rng = np.random.default_rng(42)
_show_grid(rng.choice(correct, size=min(8, len(correct)), replace=False), "Correct predictions")
if len(wrong):
    _show_grid(rng.choice(wrong, size=min(8, len(wrong)), replace=False), "Misclassifications")
else:
    print("No misclassifications — perfect test accuracy!")

In [ ]:
# --- Extended analysis: threshold sweep ---
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score, average_precision_score

thresholds = np.linspace(0.05, 0.95, 37)
rows = []
for t in thresholds:
    pred_t = (all_scores >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, pred_t, average='binary', zero_division=0)
    rows.append({
        "threshold": float(t),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "balanced_acc": float(balanced_accuracy_score(all_labels, pred_t)),
    })

ts_df = pd.DataFrame(rows)
best_f1 = ts_df.sort_values("f1", ascending=False).iloc[0]
print("Best threshold by F1:")
print(best_f1.to_string())
print(f"Average precision (PR-AUC): {average_precision_score(all_labels, all_scores):.4f}")
display(ts_df.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
for metric in ["precision", "recall", "f1", "balanced_acc"]:
    ax.plot(ts_df["threshold"], ts_df[metric], label=metric)
ax.axvline(best_f1["threshold"], color="black", ls="--", lw=1, label="best F1 threshold")
ax.set_title("Metric sensitivity to classification threshold")
ax.set_xlabel("threshold")
ax.set_ylabel("score")
ax.legend()
plt.tight_layout()


In [ ]:
# --- Extended analysis: per-defect-type breakdown ---
from pathlib import Path as _P
err_df = pd.DataFrame({
    "path": all_paths,
    "true": all_labels,
    "pred": all_preds,
    "score": all_scores,
})
err_df["defect_type"] = err_df["path"].apply(lambda p: _P(p).parent.name)
err_df["is_correct"] = (err_df["true"] == err_df["pred"])

per_type = err_df.groupby("defect_type").agg(
    n=("path", "count"),
    accuracy=("is_correct", "mean"),
    mean_score=("score", "mean"),
    true_rate=("true", "mean"),
).sort_values("n", ascending=False)

display(per_type)

fig, ax = plt.subplots(figsize=(10, 4))
plot = per_type.sort_values("accuracy")
ax.bar(plot.index, plot["accuracy"], color="teal", alpha=0.8)
ax.set_ylim(0, 1)
ax.set_title(f"{CATEGORY}: accuracy by defect type")
ax.tick_params(axis='x', rotation=60)
plt.tight_layout()


In [ ]:
# --- Extended analysis: high-confidence errors and uncertainty ---
eda = pd.DataFrame({
    "path": all_paths,
    "true": all_labels,
    "pred": all_preds,
    "score": all_scores,
})
eda["confidence"] = np.where(eda["pred"] == 1, eda["score"], 1 - eda["score"])
eda["correct"] = eda["true"] == eda["pred"]

high_conf_errors = eda[(~eda["correct"])].sort_values("confidence", ascending=False).head(15)
most_uncertain = eda.assign(uncertainty=np.abs(eda["score"] - 0.5)).sort_values("uncertainty").head(15)

print("High-confidence errors:")
display(high_conf_errors)
print("Most uncertain predictions:")
display(most_uncertain)
